In [5]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from Functions.Utils import *
from Functions.Graphs import *
random.seed(101)
# --- 1. SETUP HYPERPARAMETERS ---
N_input = 40     # Widened lookback window (~5.8 minutes of historical context)
N_p = 5          # 5 steps ahead * 7s = 35 seconds prediction horizon
batch_size = 256  
epochs = 80      
hidden_n = 128
num_layers = 3   # Kept at 2 layers to prevent gradient sinking
learning_rate = 0.0005 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Dummy definitions for functions used in your pipeline 
# (Replace with your actual implementations)
def exponential_moving_average_df(df, alpha=0.3):
    return df.ewm(alpha=alpha, adjust=False).mean()

# --- 2. LOAD SEPARATE TRIP LISTS ---
path_train = r'Datasets\DEVRT\NISSAN_LEAF_TRAIN'
samples_train = [os.path.join(path_train, s) for s in os.listdir(path_train)]

path_test_dir = r'Datasets\DEVRT\NISSAN_LEAF_TEST'
all_test_samples = os.listdir(path_test_dir)
samples_test_names = random.sample(all_test_samples, int(len(all_test_samples) * 0.5))
samples_val_names = list(set(all_test_samples) - set(samples_test_names))

samples_test = [os.path.join(path_test_dir, s) for s in samples_test_names]
samples_val = [os.path.join(path_test_dir, s) for s in samples_val_names]

def load_trip_data(file_paths):
    trips = []
    for path in file_paths:
        df = pd.read_csv(path)
        df = df.ffill().bfill()
        #df = pd.concat([df] * 3, ignore_index=True)
        speed_raw = df['speed'].values
        speed_delta = np.diff(speed_raw, prepend=speed_raw[0])
        accel_trend = pd.Series(speed_delta).rolling(window=3, min_periods=1).mean().values
        
        alt_raw = df['altitude'].values
        alt_delta = np.diff(alt_raw, prepend=alt_raw[0])
        
        trip_df = pd.DataFrame({
            'speed': speed_raw, 
            #'speed_delta': speed_delta,   
            #'accel_trend': accel_trend,   
            'rpm': df['rpm'].values, 
            'torque': df['Torque Nm'].values, 
            #'elv': df['elv_spy'].values,
            #'alt_delta': alt_delta,       
            #'max_speed': df['max_speed'].values,       
            #'soc': df['soc'].values,                   
            #'frontal_wind': df['Frontal_Wind'].values, 
            #'aux_pwr': df['Aux Pwr(100w)'].values,     
            'power': df['Motor Pwr(w)'].values,
            #'power': exponential_moving_average(df['Motor Pwr(w)'],alpha=0.5).values
        })
        trip_df = moving_average_df(trip_df,4)
        trip_df = exponential_moving_average_df(trip_df,0.25)
        trips.append(trip_df)
    return trips

print("Loading dataset trips...")
trips_train = load_trip_data(samples_train)
trips_val = load_trip_data(samples_val)
trips_test = load_trip_data(samples_test)

# --- 3. FIT SCALERS ON TRAINING DATA ONLY ---
df_all_train = pd.concat(trips_train, axis=0)
min_power = df_all_train['power'].min()
power_shift = abs(min_power) if min_power < 0 else 0

# Apply log transform to target power across all trip matrices
for t_list in [trips_train, trips_val, trips_test]:
    for trip in t_list:
        trip['power'] = np.log1p(trip['power'] + power_shift)

df_all_train_transformed = pd.concat(trips_train, axis=0)
scaler_X = StandardScaler().fit(df_all_train_transformed.values[:, :-1])
num_features = df_all_train_transformed.shape[1] - 1

# --- 4. WINDOW SLIDING FUNCTION ---
def create_dataset_sequences(trips_list, scaler_x, n_in, n_out):
    X, Y = [], []
    for trip in trips_list:
        data = trip.values
        if len(data) < (n_in + n_out):
            continue
        features_scaled = scaler_x.transform(data[:, :-1])
        power_log = data[:, -1]
        full_features = np.hstack((features_scaled, power_log.reshape(-1, 1)))
        
        for k in range(len(data) - n_in - n_out + 1):
            X.append(full_features[k : k + n_in])
            Y.append(power_log[k + n_in : k + n_in + n_out])
            
    return np.array(X), np.array(Y)

print("Building windowed timeline tensors...")
X_train, Y_train = create_dataset_sequences(trips_train, scaler_X, N_input, N_p)
X_val, Y_val = create_dataset_sequences(trips_val, scaler_X, N_input, N_p)
X_test, Y_test = create_dataset_sequences(trips_test, scaler_X, N_input, N_p)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(Y_train, dtype=torch.float32)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(Y_val, dtype=torch.float32)), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(Y_test, dtype=torch.float32)), batch_size=batch_size, shuffle=False)

# --- 5. DEFINE MODEL WITH STATE PASSING ABILITY ---
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super(LSTMForecaster, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=num_layers, batch_first=True, dropout=0.4)
        self.ln = nn.LayerNorm(hidden_dim)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x, states=None):
        # Allow passing custom memory hidden states during streaming inference
        if states is not None:
            lstm_out, (h_n, c_n) = self.lstm(x, states)
        else:
            lstm_out, (h_n, c_n) = self.lstm(x)
            
        last_time_step = lstm_out[:, -1, :] 
        normalized_features = self.ln(last_time_step)
        predictions = self.fc(self.drop(normalized_features))
        return predictions, (h_n, c_n)

model = LSTMForecaster(input_dim=num_features + 1, hidden_dim=hidden_n, output_dim=N_p, num_layers=num_layers).to(device)
#criterion = nn.HuberLoss(delta=2.0)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5) 
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

# --- 6. TRAINING & VALIDATION LOOP WITH CHECKPOINTING ---
print("Starting training...")
best_val_loss = float('inf')

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        predictions, _ = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_x.size(0)
    avg_train_loss = train_loss / len(X_train)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            predictions, _ = model(batch_x)
            loss = criterion(predictions, batch_y)
            val_loss += loss.item() * batch_x.size(0)
    avg_val_loss = val_loss / len(X_val)
    
    scheduler.step(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_lstm_model.pth')
        checkpoint_msg = "--> Best Model Saved!"
    else:
        checkpoint_msg = ""
        
    print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} {checkpoint_msg}")

# --- 7. RELOAD OPTIMAL WEIGHTS FOR EVALUATION ---
print("\nReloading best historical model configurations...")
model.load_state_dict(torch.load('best_lstm_model.pth', weights_only=True))
model.eval()



Loading dataset trips...
Building windowed timeline tensors...
[[-1.42592845e+00 -1.37501730e+00 -9.34294383e-01]
 [-1.36587923e+00 -1.31910298e+00 -8.05822201e-01]
 [-1.25355590e+00 -1.21524559e+00 -7.38929731e-01]
 [-1.15327419e+00 -1.15702076e+00 -7.90668763e-01]
 [-1.12970132e+00 -1.15849371e+00 -8.02426261e-01]
 [-1.16933250e+00 -1.21340440e+00 -8.87554930e-01]
 [-1.26634230e+00 -1.31650907e+00 -9.21939765e-01]
 [-1.36276725e+00 -1.39222021e+00 -1.12787923e+00]
 [-1.44134518e+00 -1.46384859e+00 -1.20022755e+00]
 [-1.49812701e+00 -1.51580811e+00 -1.25593773e+00]
 [-1.53406299e+00 -1.54992568e+00 -1.20209069e+00]
 [-1.57255538e+00 -1.59261163e+00 -7.78220776e-01]
 [-1.58323386e+00 -1.60195422e+00 -5.43390580e-01]
 [-1.56229392e+00 -1.57554542e+00 -2.56665942e-01]
 [-1.49729774e+00 -1.50375237e+00 -1.78788252e-01]
 [-1.40454059e+00 -1.39991393e+00 -3.53175441e-01]
 [-1.28235632e+00 -1.27741340e+00 -3.39555373e-01]
 [-1.07766127e+00 -1.09857621e+00 -1.65610737e-01]
 [-8.30838757e-01 -

LSTMForecaster(
  (lstm): LSTM(4, 128, num_layers=3, batch_first=True, dropout=0.4)
  (ln): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  (drop): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)

In [2]:
# --- 8. ROW-BY-ROW ITERATIVE STREAMING FUNCTION DEFINITION ---
# Initialize persistent variables for streaming tracking
h_step = torch.zeros(num_layers, 1, hidden_n, device=device)
c_step = torch.zeros(num_layers, 1, hidden_n, device=device)
last_predicted_log_power = 0.0

def pred_single_row(features_raw):
    """
    Accepts a single row of features at a single timestamp, scales it, 
    injects the last known power feedback, and updates the global LSTM states.
    """
    global h_step, c_step, last_predicted_log_power
    
    # 1. Shape to 2D array for scaler compatibility -> [1, num_features]
    x_arr = np.array(features_raw, dtype=np.float32).reshape(1, -1)
    x_scaled = scaler_X.transform(x_arr)
    
    # 2. Append the target layer prediction from the previous step
    full_x_row = np.hstack((x_scaled, np.array([[last_predicted_log_power]], dtype=np.float32)))
    
    # 3. Shape into standard PyTorch Recurrent structure -> [Batch=1, Time_Step=1, Features]
    x_tensor = torch.tensor(full_x_row, dtype=torch.float32).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # 4. Process step and capture updated context tuple
        pred_scaled, (h_step, c_step) = model(x_tensor, (h_step, c_step))
        pred_log = pred_scaled.cpu().numpy().flatten()
        
    # 5. Buffer the current output for the next row's past feature assignment
    last_predicted_log_power = pred_log[0]
    
    # 6. Revert Log Transform and Shift back into physical Watts
    predicted_power = np.expm1(pred_log) - power_shift
    return np.clip(predicted_power, 0, None)

# --- 9. STREAMING SIMULATION PIPELINE RUN ---
# Select the first raw test file path to simulate real-time processing
stream_file_path = samples_val[1]
df_stream = pd.read_csv(stream_file_path).ffill().bfill()

# Reconstruct features identically to the loading loop parameters
speed_raw = df_stream['speed'].values
speed_delta = np.diff(speed_raw, prepend=speed_raw[0])
accel_trend = pd.Series(speed_delta).rolling(window=3, min_periods=1).mean().values
alt_raw = df_stream['altitude'].values
alt_delta = np.diff(alt_raw, prepend=alt_raw[0])

trip_df_stream = pd.DataFrame({
            'speed': speed_raw, 
            #'speed_delta': speed_delta,   
            #'accel_trend': accel_trend,   
            'rpm': df_stream['rpm'].values, 
            'torque': df_stream['Torque Nm'].values, 
            #'elv': df_stream['elv_spy'].values,
            #'alt_delta': alt_delta,       
            #'max_speed': df['max_speed'].values,       
            #'soc': df['soc'].values,                   
            #'frontal_wind': df['Frontal_Wind'].values, 
            #'aux_pwr': df['Aux Pwr(100w)'].values,     
            'power': df_stream['Motor Pwr(w)'].values,
            #'power': exponential_moving_average(df['Motor Pwr(w)'],alpha=0.5).values
        })

trip_df_stream = moving_average_df(trip_df_stream, 4)
trip_df_stream = exponential_moving_average_df(trip_df_stream, 0.25)

# Extract inputs and real targets
stream_features = trip_df_stream.drop(columns=['power']).values
#actual_power_profile = df_stream['Motor Pwr(w)'].values
actual_power_profile = trip_df_stream['power'].values

print(f"\n==============================")
print(f"Simulating Sequential Processing Over {len(stream_features)} Rows...")
print(f"==============================\n")

# Reset memory vectors before beginning the trip
h_step = torch.zeros(num_layers, 1, hidden_n, device=device)
c_step = torch.zeros(num_layers, 1, hidden_n, device=device)
last_predicted_log_power = 0.0

yR, yP = [], []

for idx in range(len(stream_features)-N_p):
    current_timestamp_features = stream_features[idx]
    
    # Dynamic Step Call
    horizon_forecast = pred_single_row(current_timestamp_features)
    horizon_real = actual_power_profile[idx : idx + N_p]
    if idx == 0:
        yR = horizon_real.reshape(-1,N_p)
        yP = horizon_forecast.reshape(-1,N_p)
    else:
        yR = np.vstack((yR,horizon_real))
        yP = np.vstack((yP,horizon_forecast))

yR = yR.T
yP = yP.T

i=3
PlotSeriesPLY(ySeries=[yR[i],yP[i]])


Simulating Sequential Processing Over 129 Rows...

